In [1]:
import os
import re
from collections import defaultdict
from nltk.stem import PorterStemmer

# Initialize
stemmer = PorterStemmer()
word_dict = {}
doc_dict = {}
token_id = 1
doc_id = 1

# Load stopwords
def load_stop_words(path):
    with open(path, 'r') as f:
        return {line.strip() for line in f}

stop_words = load_stop_words('stopwordlist.txt')

# Tokenization + Stemming
def tokenize_and_stem(text):
    tokens = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    tokens = [t for t in tokens if t not in stop_words]
    return [stemmer.stem(t) for t in tokens]

# Parser + Dictionary Builder
def parse_documents(folder):
    global token_id, doc_id
    forward_index = {}
    inverted_index = defaultdict(dict)

    files = os.listdir(folder)
    for filename in files:
        filepath = os.path.join(folder, filename)
        with open(filepath, 'r') as file:
            content = file.read()
            documents = re.findall(r'<DOC>(.*?)</DOC>', content, re.DOTALL)

            for doc in documents:
                doc_no = re.search(r'<DOCNO>(.*?)</DOCNO>', doc).group(1).strip()
                if doc_no not in doc_dict:
                    doc_dict[doc_no] = doc_id
                    doc_id += 1
                current_doc_id = doc_dict[doc_no]

                text_match = re.search(r'<TEXT>(.*?)</TEXT>', doc, re.DOTALL)
                if not text_match:
                    continue
                text = text_match.group(1).strip()
                tokens = tokenize_and_stem(text)

                forward_index[current_doc_id] = {}
                for token in tokens:
                    if token not in word_dict:
                        word_dict[token] = token_id
                        token_id += 1
                    tid = word_dict[token]
                    forward_index[current_doc_id][tid] = forward_index[current_doc_id].get(tid, 0) + 1

                    if tid not in inverted_index:
                        inverted_index[tid] = {}
                    inverted_index[tid][current_doc_id] = inverted_index[tid].get(current_doc_id, 0) + 1
    return forward_index, inverted_index

# Save Index in Correct Format
def save_index(index, filename):
    with open(filename, 'w') as f:
        for key, values in index.items():
            line = f"{key}: " + "; ".join([f"{k}: {v}" for k, v in values.items()])
            f.write(line + "\n")

# Main Execution
if __name__ == "__main__":
    folder = 'ft911'  # Ensure this folder path is correct
    forward_index, inverted_index = parse_documents(folder)
    save_index(forward_index, 'forward_index.txt')
    save_index(inverted_index, 'inverted_index.txt')
    print("Forward and Inverted Index generated in required format.")


Forward and Inverted Index generated in required format.
